# Using the FunctionMultiplexer Class in baseobjects

## Introduction

The `FunctionMultiplexer` class is a specialized version of `CallableMultiplexer` designed specifically for working with standalone functions. It provides a mechanism for selecting between different functions at runtime without binding them to any instance. This makes it ideal for scenarios where you need to dynamically switch between different function implementations.

This tutorial will guide you through:
- Understanding the purpose and functionality of the `FunctionMultiplexer` class
- Creating and using a function multiplexer
- Adding functions to the multiplexer
- Selecting which function to use at runtime
- Understanding how `FunctionMultiplexer` differs from `CallableMultiplexer` and `MethodMultiplexer`
- Practical use cases for function multiplexers

**Prerequisites:**
- Basic understanding of Python functions
- Familiarity with callable objects in Python
- Understanding of function objects in Python

### Table of Contents

- [Importing the Module](#Importing-the-Module)
- [Core Functionality](#Core-Functionality)
- [Module Interaction](#Module-Interaction)
- [Advanced Features](#Advanced-Features)
- [Examples](#Examples)
- [API Highlights](#API-Highlights)
- [Troubleshooting / FAQs](#Troubleshooting-/-FAQs)
- [Conclusion and Next Steps](#Conclusion-and-Next-Steps)

## Importing the Module

In [1]:
from baseobjects.functions import FunctionMultiplexer, FunctionRegistry

## Core Functionality

The `FunctionMultiplexer` class is designed to select between different standalone functions to be used as a callable. It provides a way to dynamically switch between different function implementations at runtime. Let's explore the core functionality and understand how it works.

### Basic Concept

At its core, `FunctionMultiplexer` is a callable object that:

1. Maintains a registry of functions
2. Selects a specific function to use when called
3. Does not bind functions to instances (unlike `MethodMultiplexer`)
4. Optimized for working with standalone functions

Let's start with a simple example of creating and using a `FunctionMultiplexer`:

In [2]:
# Create a simple function registry
registry = FunctionRegistry()

# Define some functions to add to the registry
def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

def multiply(a, b):
    return a * b

def divide(a, b):
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b

# Add functions to the registry
registry['add'] = add
registry['subtract'] = subtract
registry['multiply'] = multiply
registry['divide'] = divide

# Create a FunctionMultiplexer with the registry
multiplexer = FunctionMultiplexer(registry=registry)

# Select a function to use
multiplexer.select('add')

# Use the multiplexer as a callable
result = multiplexer(5, 3)
print(f"5 + 3 = {result}")

# Change the selected function
multiplexer.select('multiply')
result = multiplexer(5, 3)
print(f"5 * 3 = {result}")

# Try division
multiplexer.select('divide')
result = multiplexer(10, 2)
print(f"10 / 2 = {result}")

# Handle division by zero
try:
    result = multiplexer(10, 0)
except ValueError as e:
    print(f"Error: {e}")

5 + 3 = 8
5 * 3 = 15
10 / 2 = 5.0
Error: Cannot divide by zero


In the example above, we created a simple function registry and added four functions to it. We then created a `FunctionMultiplexer` with this registry and selected which function to use. The multiplexer can be called directly, and it will delegate the call to the selected function.

### Adding and Selecting Functions

You can add functions to the registry in several ways:

In [3]:
# Create a new registry and multiplexer
registry = FunctionRegistry()
multiplexer = FunctionMultiplexer(registry=registry)

# 1. Add functions to the registry directly
registry['square'] = lambda x: x ** 2
registry['cube'] = lambda x: x ** 3

# 2. Add functions using the add_function method
def double(x):
    return x * 2

def triple(x):
    return x * 3

multiplexer.add_function('double', double)
multiplexer.add_function('triple', triple)

# 3. Add and select a function in one step
def quadruple(x):
    return x * 4

multiplexer.add_select_function('quadruple', quadruple)

# Test the functions
print(f"quadruple(5) = {multiplexer(5)}")  # Using the currently selected function

multiplexer.select('square')
print(f"square(5) = {multiplexer(5)}")

multiplexer.select('cube')
print(f"cube(5) = {multiplexer(5)}")

multiplexer.select('double')
print(f"double(5) = {multiplexer(5)}")

multiplexer.select('triple')
print(f"triple(5) = {multiplexer(5)}")

quadruple(5) = 20
square(5) = 25
cube(5) = 125
double(5) = 10
triple(5) = 15


### Working with Lambda Functions and Closures

`FunctionMultiplexer` works well with lambda functions and closures, making it flexible for various use cases:

In [4]:
# Create a multiplexer with lambda functions
registry = FunctionRegistry()
multiplexer = FunctionMultiplexer(registry=registry)

# Add lambda functions
registry['add'] = lambda a, b: a + b
registry['multiply'] = lambda a, b: a * b

# Create a closure
def create_power_function(exponent):
    def power_function(x):
        return x ** exponent
    return power_function

# Add closures to the registry
registry['square'] = create_power_function(2)
registry['cube'] = create_power_function(3)
registry['fourth_power'] = create_power_function(4)

# Test the functions
multiplexer.select('add')
print(f"add(3, 4) = {multiplexer(3, 4)}")

multiplexer.select('square')
print(f"square(5) = {multiplexer(5)}")

multiplexer.select('fourth_power')
print(f"fourth_power(2) = {multiplexer(2)}")

add(3, 4) = 7
square(5) = 25
fourth_power(2) = 16


### Difference from MethodMultiplexer

Unlike `MethodMultiplexer`, `FunctionMultiplexer` does not bind functions to instances. This means it's not suitable for working with methods that expect a `self` parameter. Let's see the difference:

In [5]:
# Create a class with methods
class MathOperations:
    def add(self, a, b):
        return a + b
    
    def subtract(self, a, b):
        return a - b

# Create an instance of the class
math_ops = MathOperations()

# Create a registry and add methods from the instance
registry = FunctionRegistry()
registry['add'] = math_ops.add  # This is already bound to math_ops
registry['subtract'] = math_ops.subtract  # This is already bound to math_ops

# Create a FunctionMultiplexer with the registry
multiplexer = FunctionMultiplexer(registry=registry)

# Select a method to use
multiplexer.select('add')

# This works because the methods are already bound to the instance
result = multiplexer(5, 3)
print(f"5 + 3 = {result}")

# But if we try to use unbound methods, it will fail
registry = FunctionRegistry()
registry['add'] = MathOperations.add  # Unbound method
multiplexer = FunctionMultiplexer(registry=registry)
multiplexer.select('add')

try:
    result = multiplexer(5, 3)
    print(f"5 + 3 = {result}")
except TypeError as e:
    print(f"Error: {e}")

5 + 3 = 8
Error: MathOperations.add() missing 1 required positional argument: 'b'


In this example, we see that `FunctionMultiplexer` works with methods that are already bound to an instance (like `math_ops.add`), but it doesn't work with unbound methods (like `MathOperations.add`) because it doesn't bind them to an instance. For working with unbound methods, you should use `MethodMultiplexer` instead.

## Module Interaction

The `FunctionMultiplexer` class is part of the `baseobjects.functions` module and interacts with other components of the baseobjects package. It extends the `CallableMultiplexer` class and specializes it for working with standalone functions.

Let's see how `FunctionMultiplexer` interacts with other components:

In [6]:
# Import necessary components
from baseobjects.functions import CallableMultiplexer, MethodMultiplexer

# Create a custom processor that uses different multiplexers
class MultiprocessingSystem:
    def __init__(self):
        # Create a registry for our functions
        self.registry = FunctionRegistry()
        
        # Add some functions to the registry
        self.registry['add'] = lambda a, b: a + b
        self.registry['subtract'] = lambda s, a, b: a - b
        self.registry['multiply'] = lambda s, a, b: a * b
        self.registry['divide'] = lambda s, a, b: a / b if b != 0 else float('inf')
        
        # Create different types of multiplexers
        self.function_multiplexer = FunctionMultiplexer(registry=self.registry)
        self.callable_multiplexer = CallableMultiplexer(registry=self.registry, instance=self)
        self.method_multiplexer = MethodMultiplexer(registry=self.registry, instance=self)
        
        # Set default functions
        self.function_multiplexer.select('add')
        self.callable_multiplexer.select('add')
        self.method_multiplexer.select('add')
    
    def process_with_function(self, a, b):
        """Process using the FunctionMultiplexer."""
        return self.function_multiplexer(a, b)
    
    def process_with_callable(self, a, b):
        """Process using the CallableMultiplexer."""
        return self.callable_multiplexer(a, b)
    
    def process_with_method(self, a, b):
        """Process using the MethodMultiplexer."""
        return self.method_multiplexer(a, b)
    
    def set_operation(self, operation_name):
        """Set the operation for all multiplexers."""
        self.function_multiplexer.select(operation_name)
        self.callable_multiplexer.select(operation_name)
        self.method_multiplexer.select(operation_name)

# Create a multiprocessing system
system = MultiprocessingSystem()

# Test with different multiplexers
a, b = 10, 5
print(f"Call a function that has a normal pattern")
print(f"FunctionMultiplexer: {a} + {b} = {system.process_with_function(a, b)}")

# CallableMultiplexer needs is_binding_wrapper=False for standalone functions
system.callable_multiplexer.is_binding_wrapper = False
print(f"CallableMultiplexer: {a} + {b} = {system.process_with_callable(a, b)}")

try:
    print(f"MethodMultiplexer: {a} + {b} = {system.process_with_method(a, b)}")
except TypeError as e:
    print(f"MethodMultiplexer cannot be use on functions which do not have a method pattern: {e}")

# Change the operation and test again
system.set_operation('multiply')
print("\nCall a function that has a method pattern")
try:
    print(f"FunctionMultiplexer: {a} * {b} = {system.process_with_function(a, b)}")
except TypeError as e:
    print(f"FunctionMultiplexer cannot be use on functions which do not have a normal function pattern: {e}")

# CallableMultiplexer needs is_binding_wrapper=True for methods
system.callable_multiplexer.is_binding_wrapper = True
print(f"CallableMultiplexer: {a} * {b} = {system.process_with_callable(a, b)}")

print(f"MethodMultiplexer: {a} * {b} = {system.process_with_method(a, b)}")

Call a function that has a normal pattern
FunctionMultiplexer: 10 + 5 = 15
CallableMultiplexer: 10 + 5 = 15
MethodMultiplexer cannot be use on function which do not have a method pattern: MultiprocessingSystem.__init__.<locals>.<lambda>() takes 2 positional arguments but 3 were given
Call a function that has a method pattern

After changing to multiply:
FunctionMultiplexer cannot be use on method which do not have a function pattern: MultiprocessingSystem.__init__.<locals>.<lambda>() missing 1 required positional argument: 'b'
CallableMultiplexer: 10 * 5 = 50
MethodMultiplexer: 10 * 5 = 50


In this example, we created a system that uses all three types of multiplexers. We can see that `FunctionMultiplexer` works seamlessly with standalone functions, while `CallableMultiplexer` needs `is_binding_wrapper=False` to work with standalone functions, and `MethodMultiplexer` works with methods.

## Advanced Features

### Creating a Custom FunctionMultiplexer

You can create a custom `FunctionMultiplexer` by subclassing it and adding your own functionality:

In [7]:
# Create a custom FunctionMultiplexer that logs function calls
class LoggingFunctionMultiplexer(FunctionMultiplexer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.call_log = []
    
    def __call__(self, *args, **kwargs):
        """Override __call__ to log function calls."""
        # Get the name of the selected function
        function_name = self.selected
        
        # Log the function call
        self.call_log.append({
            'function': function_name,
            'args': args,
            'kwargs': kwargs
        })
        
        # Call the function using the parent's __call__ method
        result = super().__call__(*args, **kwargs)
        
        # Log the result
        self.call_log[-1]['result'] = result
        
        return result
    
    def show_log(self):
        """Show the function call log."""
        print("Function Call Log:")
        for i, entry in enumerate(self.call_log, 1):
            args_str = ', '.join(str(arg) for arg in entry['args'])
            kwargs_str = ', '.join(f"{k}={v}" for k, v in entry['kwargs'].items())
            all_args = ', '.join(filter(None, [args_str, kwargs_str]))
            print(f"{i}. {entry['function']}({all_args}) => {entry['result']}")

# Create a registry with functions
registry = FunctionRegistry()
registry['add'] = lambda a, b: a + b
registry['multiply'] = lambda a, b: a * b
registry['square'] = lambda x: x ** 2

# Create a logging function multiplexer
multiplexer = LoggingFunctionMultiplexer(registry=registry)

# Use the multiplexer with different functions
multiplexer.select('add')
result1 = multiplexer(5, 3)

multiplexer.select('multiply')
result2 = multiplexer(5, 3)

multiplexer.select('square')
result3 = multiplexer(5)

# Show the log
multiplexer.show_log()

Function Call Log:
1. add(5, 3) => 8
2. multiply(5, 3) => 15
3. square(5) => 25


### Function Selection Based on Input Types

Another advanced use case is selecting functions based on the types of the input arguments:

In [8]:
# Create a function multiplexer that selects functions based on input types
class TypeBasedFunctionMultiplexer(FunctionMultiplexer):
    def __call__(self, *args, **kwargs):
        """Select a function based on the types of the input arguments."""
        # Determine the types of the arguments
        if args and isinstance(args[0], str):
            self.select('string_processor')
        elif args and isinstance(args[0], (int, float)):
            self.select('number_processor')
        elif args and isinstance(args[0], list):
            self.select('list_processor')
        elif args and isinstance(args[0], dict):
            self.select('dict_processor')
        else:
            self.select('default_processor')
        
        # Call the selected function
        return super().__call__(*args, **kwargs)

# Create functions for different types
def string_processor(text):
    return f"Processing string: {text.upper()}"

def number_processor(num):
    return f"Processing number: {num * 2}"

def list_processor(items):
    return f"Processing list: {sorted(items)}"

def dict_processor(data):
    return f"Processing dict: {list(data.keys())}"

def default_processor(*args):
    return f"Processing unknown type: {args}"

# Create a registry and add the functions
registry = FunctionRegistry()
registry['string_processor'] = string_processor
registry['number_processor'] = number_processor
registry['list_processor'] = list_processor
registry['dict_processor'] = dict_processor
registry['default_processor'] = default_processor

# Create a type-based function multiplexer
multiplexer = TypeBasedFunctionMultiplexer(registry=registry)

# Test with different types of inputs
print(multiplexer("hello"))
print(multiplexer(42))
print(multiplexer([3, 1, 4, 1, 5, 9]))
print(multiplexer({"name": "John", "age": 30}))
print(multiplexer(None))

Processing string: HELLO
Processing number: 84
Processing list: [1, 1, 3, 4, 5, 9]
Processing dict: ['name', 'age']
Processing unknown type: (None,)


## Examples

### Example 1: Function-Based Calculator

Let's implement a simple calculator using `FunctionMultiplexer`:

In [9]:
# Create a calculator using FunctionMultiplexer
class Calculator:
    def __init__(self):
        # Create a registry for our operations
        self.registry = FunctionRegistry()
        
        # Add basic operations
        self.registry['add'] = lambda a, b: a + b
        self.registry['subtract'] = lambda a, b: a - b
        self.registry['multiply'] = lambda a, b: a * b
        self.registry['divide'] = lambda a, b: a / b if b != 0 else float('inf')
        self.registry['power'] = lambda a, b: a ** b
        self.registry['modulo'] = lambda a, b: a % b
        
        # Create a FunctionMultiplexer with our registry
        self.operation = FunctionMultiplexer(registry=self.registry)
        
        # Set a default operation
        self.operation.select('add')
    
    def calculate(self, a, b):
        """Perform the selected operation on the given numbers."""
        return self.operation(a, b)
    
    def set_operation(self, operation_name):
        """Set the operation to use."""
        if operation_name not in self.registry:
            raise ValueError(f"Unknown operation: {operation_name}")
        
        self.operation.select(operation_name)
        return f"Operation set to: {operation_name}"

# Create a calculator
calculator = Calculator()

# Perform calculations with different operations
a, b = 10, 3
print(f"Default operation (add): {a} + {b} = {calculator.calculate(a, b)}")

calculator.set_operation('multiply')
print(f"Multiply: {a} * {b} = {calculator.calculate(a, b)}")

calculator.set_operation('power')
print(f"Power: {a} ^ {b} = {calculator.calculate(a, b)}")

calculator.set_operation('divide')
print(f"Divide: {a} / {b} = {calculator.calculate(a, b)}")

calculator.set_operation('modulo')
print(f"Modulo: {a} % {b} = {calculator.calculate(a, b)}")

# Add a new operation dynamically
calculator.registry['absolute_difference'] = lambda a, b: abs(a - b)
calculator.set_operation('absolute_difference')
print(f"Absolute difference: |{a} - {b}| = {calculator.calculate(a, b)}")

Default operation (add): 10 + 3 = 13
Multiply: 10 * 3 = 30
Power: 10 ^ 3 = 1000
Divide: 10 / 3 = 3.3333333333333335
Modulo: 10 % 3 = 1
Absolute difference: |10 - 3| = 7


### Example 2: Text Processing Pipeline

Let's implement a text processing pipeline using `FunctionMultiplexer`:

In [10]:
# Create a text processing pipeline using FunctionMultiplexer
class TextProcessor:
    def __init__(self):
        # Create a registry for our text processing functions
        self.registry = FunctionRegistry()
        
        # Add text processing functions
        self.registry['uppercase'] = lambda text: text.upper()
        self.registry['lowercase'] = lambda text: text.lower()
        self.registry['capitalize'] = lambda text: text.title()
        self.registry['reverse'] = lambda text: text[::-1]
        self.registry['remove_spaces'] = lambda text: text.replace(' ', '')
        self.registry['count_chars'] = lambda text: f"{text} ({len(text)} characters)"
        
        # Create a FunctionMultiplexer with our registry
        self.processor = FunctionMultiplexer(registry=self.registry)
        
        # Create a pipeline of processors
        self.pipeline = []
    
    def add_to_pipeline(self, processor_name):
        """Add a processor to the pipeline."""
        if processor_name not in self.registry:
            raise ValueError(f"Unknown processor: {processor_name}")
        
        self.pipeline.append(processor_name)
        return f"Added {processor_name} to pipeline"
    
    def clear_pipeline(self):
        """Clear the pipeline."""
        self.pipeline = []
        return "Pipeline cleared"
    
    def process(self, text):
        """Process the text through the pipeline."""
        result = text
        for processor_name in self.pipeline:
            self.processor.select(processor_name)
            result = self.processor(result)
        return result

# Create a text processor
processor = TextProcessor()

# Set up a pipeline
text = "hello world"
processor.add_to_pipeline('uppercase')
processor.add_to_pipeline('reverse')
processor.add_to_pipeline('count_chars')

# Process the text
result = processor.process(text)
print(f"Original text: '{text}'")
print(f"Processed text: '{result}'")

# Change the pipeline
processor.clear_pipeline()
processor.add_to_pipeline('capitalize')
processor.add_to_pipeline('remove_spaces')

# Process the text again
result = processor.process(text)
print(f"\nOriginal text: '{text}'")
print(f"Processed text with new pipeline: '{result}'")

Original text: 'hello world'
Processed text: 'DLROW OLLEH (11 characters)'

Original text: 'hello world'
Processed text with new pipeline: 'HelloWorld'


## API Highlights

The `FunctionMultiplexer` class provides the following key features:

- **Function Selection**: Select between different functions at runtime
- **Registry Integration**: Use a `FunctionRegistry` to store and manage functions
- **No Instance Binding**: Does not bind functions to instances (unlike `MethodMultiplexer`)
- **Optimized for Functions**: Specialized for working with standalone functions

Key methods:
- `__init__(registry=None, instance=None, owner=None, select=None, ...)`: Constructor with options for initial setup
- `select(name)`: Select a function to use
- `add_function(name, func)`: Add a function to the registry
- `add_select_function(name, func)`: Add a function to the registry and select it

For the full API documentation, refer to the baseobjects documentation.

## Troubleshooting / FAQs

### Q: When should I use FunctionMultiplexer instead of CallableMultiplexer or MethodMultiplexer?

A: Use `FunctionMultiplexer` when you're working exclusively with standalone functions and don't need method binding. It's optimized for this use case and provides a cleaner interface for function-based operations.

### Q: Can FunctionMultiplexer work with methods?

A: `FunctionMultiplexer` can work with methods that are already bound to an instance (like `obj.method`), but it doesn't bind unbound methods to instances. If you need to work with unbound methods, use `MethodMultiplexer` instead.

### Q: How do I check which function is currently selected?

A: You can check the currently selected function using the `selected` property:

```python
multiplexer = FunctionMultiplexer(registry=registry)
multiplexer.select('add')
print(f"Currently selected function: {multiplexer.selected}")
```

## Conclusion and Next Steps

In this tutorial, we've explored the `FunctionMultiplexer` class from the baseobjects package. We've learned how to use it to create a callable that can dynamically select between different functions at runtime.

Key takeaways:
- `FunctionMultiplexer` is a specialized version of `CallableMultiplexer` for working with standalone functions
- It provides a clean interface for selecting between different function implementations at runtime
- It does not bind functions to instances, making it ideal for function-based operations
- It can be extended and customized for specific use cases

Next steps:
- Explore the `MethodMultiplexer` class for working with methods
- Combine `FunctionMultiplexer` with other components of the baseobjects package
- Create your own custom function multiplexers by subclassing `FunctionMultiplexer`
- Check out the examples directory for more examples of using function multiplexers

For more information, refer to the baseobjects documentation and examples.